In [3]:
# Set project directory
%cd /content/drive/MyDrive/app_note_rag

/content/drive/MyDrive/app_note_rag


In [14]:
!pip install psycopg FlagEmbedding langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 31.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3


In [ ]:
# Initialize the database

from src.database import init_database

init_database()

✅ PostgreSQL initialized and database restored.


In [ ]:
# Import RAG components

from src.rag.retrieval.hybrid import hybrid_search
from src.rag.retrieval.reranker import reranker
from src.rag.retrieval.context import build_context

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
# Retrieve and rerank candidates

query = "artificial intelligence research"

results = hybrid_search(
    query=query,
    user_id=4,
    top_k=10,
)

reranked_results = reranker.rerank(
    query=query,
    results=results,
    top_k=3,
)

In [ ]:
# Build retrieved context

context = build_context(reranked_results)

print(context)

[Source 1]
Title: AI Research
Content:
Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.

[Source 2]
Title: Machine Learning
Content:
Machine learning is a field of artificial intelligence focused on learning
patterns from data. Supervised learning uses labeled examples, while
unsupervised learning discovers patterns without explicit labels. Model
evaluation requires appropriate metrics, validation data, and careful control
of overfitting.

[Source 3]
Title: Retrieval Augmented Generation
Content:
Retrieval augmented generation combines information retrieval with language
model generation. A RAG system retrieves relevant documents or chunks before
generating an answer. Hybrid retrieval can combine semantic search and
keyword search, followed by fusion and reranking

In [ ]:
# Test the RAG pipeline

from src.rag.pipeline import rag_pipeline


query = "artificial intelligence research"

context = rag_pipeline.retrieve(
    query=query,
    user_id=4,
    top_k=3,
    candidate_k=10,
)

print(context)

[Source 1]
Title: AI Research
Content:
Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.

[Source 2]
Title: Machine Learning
Content:
Machine learning is a field of artificial intelligence focused on learning
patterns from data. Supervised learning uses labeled examples, while
unsupervised learning discovers patterns without explicit labels. Model
evaluation requires appropriate metrics, validation data, and careful control
of overfitting.

[Source 3]
Title: Retrieval Augmented Generation
Content:
Retrieval augmented generation combines information retrieval with language
model generation. A RAG system retrieves relevant documents or chunks before
generating an answer. Hybrid retrieval can combine semantic search and
keyword search, followed by fusion and reranking

In [11]:
# Test the RAG agent tool

from src.rag.agent_tool import search_notes


result = search_notes.invoke(
    {"query": "What do my notes say about artificial intelligence research?"},
    config={"configurable": {"user_id": 4}},
)

print(result)

[Source 1]
Title: AI Research
Content:
Artificial intelligence research requires a combination of machine learning,
deep learning, natural language processing, and careful experimentation.
Researchers need reliable datasets, appropriate evaluation methods, and
reproducible experiments to validate their findings.

[Source 2]
Title: Machine Learning
Content:
Machine learning is a field of artificial intelligence focused on learning
patterns from data. Supervised learning uses labeled examples, while
unsupervised learning discovers patterns without explicit labels. Model
evaluation requires appropriate metrics, validation data, and careful control
of overfitting.

[Source 3]
Title: Retrieval Augmented Generation
Content:
Retrieval augmented generation combines information retrieval with language
model generation. A RAG system retrieves relevant documents or chunks before
generating an answer. Hybrid retrieval can combine semantic search and
keyword search, followed by fusion and reranking

In [1]:
# Set the Groq API key

import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [5]:
# Test the agent with a RAG question

from src.agent.graph import graph


result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What does my AI Research note say about quantum computing?",
            }
        ]
    },
    config={
        "configurable": {
            "user_id": 4,
        }
    },
)

print(result["messages"][-1].content)

I searched your notes for any mention of quantum computing in the “AI Research” note, but the note’s content does not include that topic. It covers general AI research practices such as machine learning, deep learning, natural language processing, datasets, evaluation methods, and reproducibility, but it does not discuss quantum computing.


# Final Test for Agnet

In [6]:
# Test RAG routing

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What do my notes say about artificial intelligence research?",
            }
        ]
    },
    config={"configurable": {"user_id": 4}},
)

print(result["messages"][-1].content)

Your notes contain the following information about artificial intelligence research:

1. **AI Research (Note titled “AI Research”)**  
   - AI research combines **machine learning, deep learning, natural language processing**, and careful experimentation.  
   - Key requirements include **reliable datasets**, **appropriate evaluation methods**, and **reproducible experiments** to validate findings.

2. **Machine Learning (Note titled “Machine Learning”)**  
   - Machine learning is a subset of AI focused on learning patterns from data.  
   - It distinguishes between **supervised learning** (using labeled examples) and **unsupervised learning** (discovering patterns without explicit labels).  
   - Proper model evaluation demands suitable metrics, validation data, and careful control of overfitting.

3. **Retrieval Augmented Generation (Note titled “Retrieval Augmented Generation”)**  
   - This approach merges information retrieval with language‑model generation.  
   - A RAG system f

In [11]:
# Test note creation

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Create a note titled 'The Power of Music' with content 'Music is a universal form of expression that can influence emotions, memories, and human connection. Different genres and rhythms can create different feelings, making music an important part of everyday life and culture.'",
            }
        ]
    },
    config={"configurable": {"user_id": 4}},
)

print(result["messages"][-1].content)

Your note **“The Power of Music”** has been created successfully. Let me know if you’d like to do anything else!


In [8]:
# Test note summarization

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Summarize my note titled 'The Power of Music'.",
            }
        ]
    },
    config={"configurable": {"user_id": 4}},
)

print(result["messages"][-1].content)

**Summary of “The Power of Music”:**

- Music is a universal form of expression that influences emotions, memories, and human connection.  
- Different genres and rhythms evoke distinct feelings, underscoring music’s role in everyday life and culture.


In [10]:
# Test note deletion

result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete my note titled 'The Power of Music'.",
            }
        ]
    },
    config={"configurable": {"user_id": 4}},
)

print(result["messages"][-1].content)

Your note titled **"The Power of Music"** has been deleted successfully.
